# 10 — Variational Inference
**References:** Blei, Kucukelbir & McAuliffe (2017) · Jordan et al. (1999) · Wainwright & Jordan (2008)

## Narrative thread
```
ELBO -> KL divergence -> Mean-field VI -> CAVI algorithm -> Stochastic VI -> ADVI -> VI vs MCMC
```

## 1. The variational approach

**Problem:** Posterior $p(\theta \mid \mathbf{x})$ is intractable.

**VI idea:** Approximate the posterior by finding the member of a tractable family $\mathcal{Q}$ that is closest in KL divergence:
$$q^*(\theta) = \arg\min_{q \in \mathcal{Q}} \text{KL}\!\left(q(\theta) \;\|\; p(\theta \mid \mathbf{x})\right)$$

**KL divergence:**
$$\text{KL}(q \| p) = E_q\!\left[\log\frac{q(\theta)}{p(\theta\mid\mathbf{x})}\right] \geq 0$$

**The ELBO (Evidence Lower Bound):**
$$\log p(\mathbf{x}) = \underbrace{E_q[\log p(\mathbf{x},\theta)] - E_q[\log q(\theta)]}_{\mathcal{L}(q) = \text{ELBO}} + \text{KL}(q \| p) \geq \mathcal{L}(q)$$

Minimizing $\text{KL}(q\|p) \Leftrightarrow$ maximizing the ELBO $\mathcal{L}(q)$.

The ELBO decomposes as:
$$\mathcal{L}(q) = E_q[\log p(\mathbf{x} \mid \theta)] - \text{KL}(q(\theta) \| p(\theta))$$
i.e., **expected log-likelihood minus KL to the prior**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats, optimize

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)

# ── VI for Beta-Binomial: minimize KL(q||p) over q=Beta(a,b) ─────────────
# True posterior: Beta(alpha0+x, beta0+n-x)
# Variational family: q = Beta(a, b)
# KL(Beta(a,b) || Beta(ap,bp)) has closed form

from scipy.special import digamma, betaln

def kl_beta(a1, b1, a2, b2):
    """KL(Beta(a1,b1) || Beta(a2,b2))"""
    return (betaln(a2,b2) - betaln(a1,b1) +
            (a1-a2)*digamma(a1) + (b1-b2)*digamma(b1) +
            (a2-a1+b2-b1)*digamma(a1+b1))

# True posterior
alpha0, beta0 = 2, 2; x_obs, n_obs = 3, 20
ap = alpha0 + x_obs; bp = beta0 + n_obs - x_obs

p_g = np.linspace(0.001, 0.999, 400)
true_post = stats.beta.pdf(p_g, ap, bp)

# Optimize variational parameters
def neg_elbo(params):
    a, b = np.exp(params)  # constrain positive
    return kl_beta(a, b, ap, bp)

# Backward KL (mean-seeking)
res_bkl  = optimize.minimize(neg_elbo, [1.0, 1.0], method='Nelder-Mead')
a_vi, b_vi = np.exp(res_bkl.x)
vi_post  = stats.beta.pdf(p_g, a_vi, b_vi)

# Forward KL (mode-seeking, different behavior)
def kl_forward(params):
    a, b = np.exp(params)
    return kl_beta(ap, bp, a, b)  # reversed
res_fkl  = optimize.minimize(kl_forward, [1.0, 1.0], method='Nelder-Mead')
a_fkl, b_fkl = np.exp(res_fkl.x)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(p_g, true_post, color='#4361ee', lw=2.5, label=f'True posterior Beta({ap},{bp})')
axes[0].plot(p_g, vi_post,   color='#f72585', lw=2, linestyle='--',
             label=f'VI (backward KL) Beta({a_vi:.1f},{b_vi:.1f})')
axes[0].plot(p_g, stats.beta.pdf(p_g, a_fkl, b_fkl), color='#06d6a0', lw=2, linestyle=':',
             label=f'Forward KL Beta({a_fkl:.1f},{b_fkl:.1f})')
axes[0].set_title('Backward vs Forward KL\nBoth recover true posterior here (Beta family = exact)')
axes[0].set_xlabel('p'); axes[0].legend(fontsize=7)

# KL landscape
a_range = np.linspace(0.5, 15, 80)
b_range = np.linspace(0.5, 25, 80)
AA, BB  = np.meshgrid(a_range, b_range)
KL_grid = np.vectorize(kl_beta)(AA, BB, ap, bp)
axes[1].contourf(AA, BB, np.log1p(KL_grid), levels=20, cmap='Blues_r')
axes[1].plot(a_vi, b_vi, 'r*', markersize=15, label=f'VI optimum ({a_vi:.1f},{b_vi:.1f})')
axes[1].plot(ap, bp, 'g*', markersize=15, label=f'True ({ap},{bp})')
axes[1].set_xlabel('a'); axes[1].set_ylabel('b')
axes[1].set_title('KL(q||p) landscape\nVI minimizes over (a,b) space')
axes[1].legend(fontsize=8)

# ELBO as lower bound
thetas = np.linspace(0.001, 0.999, 200)
log_joint = stats.binom.logpmf(x_obs, n_obs, thetas) + stats.beta.logpdf(thetas, alpha0, beta0)
log_marg  = np.log(np.trapz(np.exp(log_joint), thetas))

a_vals = np.linspace(0.5, 12, 100)
elbos  = []
for a in a_vals:
    b = a * bp / ap  # track true mode ratio
    # ELBO = E_q[log p(x,theta)] - E_q[log q(theta)]
    # = -KL(q||p) + log p(x)
    elbo = log_marg - kl_beta(a, b, ap, bp)
    elbos.append(elbo)

axes[2].plot(a_vals, elbos, color='#4361ee', lw=2.5, label='ELBO (varies with q)')
axes[2].axhline(log_marg, color='#f72585', lw=2, linestyle='--', label=f'log p(x)={log_marg:.3f}')
axes[2].set_xlabel('a (variational parameter)'); axes[2].set_ylabel('ELBO')
axes[2].set_title('ELBO is a lower bound on log p(x)\nMaximizing ELBO = minimizing KL')
axes[2].legend()

plt.suptitle('Variational Inference: KL Divergence and ELBO', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Mean-field variational inference

**Mean-field assumption:** $q(\boldsymbol{\theta}) = \prod_{j=1}^p q_j(\theta_j)$ — all parameters are independent in the approximation.

**CAVI (Coordinate Ascent VI):** Update each $q_j$ in turn while holding others fixed:
$$q_j^*(\theta_j) \propto \exp\!\left(E_{-j}[\log p(\mathbf{x}, \boldsymbol{\theta})]\right)$$

where $E_{-j}$ is the expectation over all $q_k$, $k \neq j$. This update is guaranteed to increase the ELBO until convergence.

**Limitation of mean-field:** Independence assumption means VI underestimates posterior correlations — posteriors are typically too narrow (overconfident). MCMC gives the true posterior; VI gives a fast approximation.

## 3. VI vs MCMC

| Aspect | MCMC | Variational Inference |
|---|---|---|
| Output | Samples from exact posterior | Approximate posterior $q^*(\theta)$ |
| Accuracy | Asymptotically exact | Biased by family choice |
| Speed | Slow for large models | Fast — scales to big data |
| Convergence | $\hat R$, ESS | ELBO convergence |
| Posterior correlations | Captured exactly | Lost (mean-field) |
| Software | Stan (NUTS) | NumPyro (SVI), PyMC (ADVI), TFP |

**ADVI (Automatic Differentiation VI):** Kucukelbir et al. (2017) — automatically derives ELBO gradient via autodiff. Powers PyMC's default fast inference.

## 4. Key takeaways

| Concept | Statement |
|---|---|
| ELBO | Lower bound on log marginal likelihood; equal when $q = p(\cdot\mid\mathbf{x})$ |
| Backward KL | Mode-seeking; tends to underestimate variance |
| Forward KL | Mean-seeking; tends to overestimate variance |
| Mean-field | Factorized approximation — fast but loses correlations |
| CAVI | Coordinate ascent updates; guaranteed ELBO increase |

**Next:** notebook 11 — Gaussian processes: nonparametric Bayesian priors over functions.